In [22]:
# Import Dependencies and Set Reproducibility Controls
import numpy as np
import pandas as pd
from pathlib import Path
import spacy

SEED = 42
np.random.seed(SEED)

# Load spacy English model for NER and text processing
nlp = spacy.load("en_core_web_sm")
print(f"Loaded spacy model: {nlp.meta['name']} v{nlp.meta['version']}")

Loaded spacy model: core_web_sm v3.8.0


In [23]:
# Configure Local Data Paths
project_root = Path("..").resolve()
raw_csv_path = project_root / "data" / "raw_batch1" / "emails.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data path:", raw_csv_path)
print("Processed dir:", processed_dir)

Project root: C:\Users\willd\Dissertation
Raw data path: C:\Users\willd\Dissertation\data\raw_batch1\emails.csv
Processed dir: C:\Users\willd\Dissertation\data\processed


In [24]:
# Load Raw Dataset
df = pd.read_csv(raw_csv_path)

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
df.head(3)

Total rows: 185
Total columns: 2


,subject,body
0,Module Registration Issue,"Dear Dr Smith,\r\n\r\nI'm trying to register f..."
1,changing modules,hi\r\n\r\ni want to drop kv4006 and switch to ...
2,Student ID Card Issue,"Dear Dr Smith,\r\n\r\nApologies for emailing y..."


In [25]:
# Validate Dataset Schema and Data Quality
required_cols = ["subject", "body"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print({
    "rows": len(df),
    "missing_subject": int(df["subject"].isna().sum()),
    "missing_body": int(df["body"].isna().sum()),
})

{'rows': 185, 'missing_subject': 0, 'missing_body': 0}


In [26]:
# Define PII Masking and Text Cleaning
import re

PII_PATTERNS = {
    "EMAIL": r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b",
    "PHONE": r"\b(?:\+44\s?7\d{3}|07\d{3})\s?\d{3}\s?\d{3}\b",
    "STUDENT_ID": r"\b(?:w|W)?\d{7,9}\b",
}

SIGN_OFF_PATTERNS = [
    r"(?im)^kind regards,?.*$",
    r"(?im)^regards,?.*$",
    r"(?im)^thanks,?.*$",
    r"(?im)^many thanks,?.*$",
]

def mask_pii(text: str) -> str:
    """Mask PII using regex patterns and spacy NER for person names."""
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    
    masked = text
    
    for label, pattern in PII_PATTERNS.items():
        masked = re.sub(pattern, f"[{label}]", masked)
    
    doc = nlp(masked)
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            masked = masked.replace(ent.text, "[PERSON]")
        elif ent.label_ == "ORG":
            masked = masked.replace(ent.text, "[ORGANIZATION]")
    
    return masked

def clean_email_text(text: str) -> str:
    """Clean email text with quoted replies, URLs, and sign-offs removed."""
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)

    cleaned = text
    cleaned = re.sub(r"(?is)\n?On .*? wrote:.*$", " ", cleaned)
    cleaned = re.sub(r"(?im)^>.*$", " ", cleaned)
    cleaned = re.sub(r"https?://\S+|www\.\S+", " [URL] ", cleaned)

    for p in SIGN_OFF_PATTERNS:
        cleaned = re.sub(p, " ", cleaned)

    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

In [27]:
### Apply PII Masking and Text Cleaning
df_email = df.copy()
df_email["subject_masked"] = df_email["subject"].apply(mask_pii)
df_email["body_masked"] = df_email["body"].apply(mask_pii)
df_email["subject_clean"] = df_email["subject_masked"].apply(clean_email_text)
df_email["body_clean"] = df_email["body_masked"].apply(clean_email_text)
df_email["text_clean"] = (df_email["subject_clean"] + " [SEP] " + df_email["body_clean"]).str.strip()

print(f"\nProcessed {len(df_email)} emails")


Processed 185 emails


In [28]:
### Save Preprocessing Results
email_path = processed_dir / "email_masked_cleaned.csv"
df_email.to_csv(email_path, index=False)
print(email_path)

C:\Users\willd\Dissertation\data\processed\email_masked_cleaned.csv


In [29]:
# load processed data and labelled source files for merging

label_dir = project_root / "data" / "batch1"
processed_path = processed_dir / "email_masked_cleaned.csv"

base_df = pd.read_csv(processed_path).copy()
label_files = sorted(label_dir.glob("*Emails.csv"))

if not label_files:
    raise FileNotFoundError(f"No prelabelled files found in {label_dir}")

print(f"Label files: {[f.name for f in label_files]}")

Label files: ['admEmails.csv', 'assEmails.csv', 'dupEmails.csv', 'extEmails.csv', 'fdbEmails.csv', 'mixEmails.csv', 'resEmails.csv', 'timEmails.csv']


In [30]:
# make one labelled dataframe with source file info for merging with base_df

label_parts = []
for f in label_files:
    d = pd.read_csv(f).copy()
    d["source_file"] = f.name
    label_parts.append(d)

labels_df = pd.concat(label_parts, ignore_index=True)
print(labels_df.shape)
labels_df.head(2)

(185, 8)


,email_id,subject,body,category,module,priority,multi_intent,source_file
0,ADM_001,Module Registration Issue,"Dear Dr Smith,\r\n\r\nI'm trying to register f...",admin,KV6003,high,False,admEmails.csv
1,ADM_002,changing modules,hi\r\n\r\ni want to drop kv4006 and switch to ...,admin,KV4006,high,False,admEmails.csv


In [31]:
# apply same masking/cleaning pipeline to labelled rows
for col in ["subject", "body"]:
    labels_df[col] = labels_df[col].fillna("").astype(str)

labels_df["subject_masked"] = labels_df["subject"].apply(mask_pii)
labels_df["body_masked"] = labels_df["body"].apply(mask_pii)
labels_df["subject_clean"] = labels_df["subject_masked"].apply(clean_email_text)
labels_df["body_clean"] = labels_df["body_masked"].apply(clean_email_text)
labels_df["text_clean"] = (labels_df["subject_clean"] + " [SEP] " + labels_df["body_clean"]).str.strip()

In [32]:
# convert category strings to label lists
def parse_label_list(value: str):
    value = "" if pd.isna(value) else str(value)
    labels = [x.strip().lower() for x in value.split("|") if x.strip()]
    return list(dict.fromkeys(labels))

labels_df["label_list"] = labels_df["category"].apply(parse_label_list)

label_map = labels_df.groupby("text_clean", as_index=False).agg({
    "label_list": lambda rows: sorted(set(label for row in rows for label in row))
})

In [35]:
# Join labels onto processed dataset
base_df = base_df.merge(label_map, on="text_clean", how="left")
base_df["label_list"] = base_df["label_list"].apply(lambda x: x if isinstance(x, list) else [])
base_df["n_labels"] = base_df["label_list"].str.len()

matched = int((base_df["n_labels"] > 0).sum())
print(f"Matched rows: {matched}/{len(base_df)} ({matched/len(base_df):.1%})")

Matched rows: 185/185 (100.0%)
